# Lab 2: Dashboard Lab
In this lab, we will build an AI/BI Dashboard to view and understand metrics about customers that are potentially going to Churn!

AI/BI Dashboards allows business users to create dashboards and visualizations from your data to enable data-driven decision-making for everyone. Users can generate detailed reports containing insights into performance, trends and key metrics to empower users to act on what their data tells them.

# What are we going to build
We are going to build a Customer churn dashboard with preditictive KPIs that helps you understand the current state of risk and identify the customers that are potentially going to churn. Here's how the dashboard is going to look. 

![](./Images/aibi_dashboard/Dashboard_Final.png)

# What you'll build in this lab

You'll build a **Customer Churn dashboard** end-to-end. You register the SQL **datasets** once (Part 1), then build the widgets. You'll build the **first widget by hand** to learn how a widget maps to a dataset — then use **Genie Code** for the rest (each Genie step also keeps the manual instructions as an optional fallback).

> **📋 Lab breakdown**
> - **Part 1 — Add Datasets:** create the dashboard and register the SQL datasets that power every widget.
> - **Part 2 — Build Widgets:** build the first widget manually, then the rest with Genie Code.
> - **Part 3 — Add a Filter:** a cross-widget Country filter.
> - **Part 4 — Publish & Schedule:** publish, schedule snapshots, manage subscribers.
> - **Part 5 — Extended Version:** more widgets (revenue-at-risk, payment-method churn, tables).
> - **Part 6 — Embed:** embed the published dashboard in an external app.
>
> 💡 From Part 2 onward, each **Genie Code prompt** already contains the full widget spec — paste it and review the result, or expand **"build it manually instead"** to do it by hand.

> **Note on data location:** these datasets point at **`data_pioneers.c360`**, which the `Assets/setup-data-pioneer` notebook creates. If your data lives in a different catalog/schema, update the table names in the Part 1 / Part 5 SQL accordingly.

##Part 1: Add Datasets


### 1.1: Create Dashboard

Navigate to the side panel menu --> SQL --> Dashboards. Click "Create Dashboard" to create a new Dashboard.

![](./Images/aibi_dashboard/Create_Dashboard.png)

### 1.2: Rename and to go Data tab

Rename the Dashboard to "{Unique prefix} Churn Prediction Dashboard" and click on "Data" to add Datasets.

![](./Images/aibi_dashboard/Dashboard_Rename.png)

### 1.3: Click on "Create from SQL" to add the first dataset

![](./Images/aibi_dashboard/Dashboard_Add_Dataset1.png)

### 1.4: Add dataset — "Churn - Customer Breakdown - Universal"

| Dataset Name | SQL Query |
| --- | --- |
| Churn - Customer Breakdown - Universal | `SELECT canal, country, count(*) as users_count FROM data_pioneers.c360.churn_features WHERE churn = 1 AND canal IS NOT NULL GROUP BY canal, country` |

Steps:
1. Rename the dataset to **Churn - Customer Breakdown - Universal**
2. Paste the SQL query
3. Click **Run** to execute the query

![](./Images/aibi_dashboard/Dashboard_Add_Dataset1_1.png)

### 1.5: Add the rest of the datasets

Repeat the "Create from SQL" step for each dataset below.

| Dataset Name | SQL Query |
| --- | --- |
| Churn - Total MRR - Universal | `SELECT sum(amount)/10 as MRR, country FROM data_pioneers.c360.churn_orders o LEFT JOIN data_pioneers.c360.churn_users cu ON o.user_id = cu.user_id WHERE (month(to_timestamp(o.creation_date, 'MM-dd-yyyy HH:mm:ss')) = (SELECT max(month(to_timestamp(creation_date, 'MM-dd-yyyy HH:mm:ss'))) FROM data_pioneers.c360.churn_orders)) GROUP BY country` |
| Churn - Predictions - Universal | `SELECT cast(days_since_creation/30 as int) as days_since_creation, churn, country, count(*) as customers FROM data_pioneers.c360.churn_features WHERE days_since_creation < 2000 GROUP BY cast(days_since_creation/30 as int), churn, country` |

> **Note:** the *Predictions* query buckets tenure into months and drops extreme-tenure outliers (`days_since_creation < 2000`) so the tenure chart's x-axis stays clean. (The classic `HAVING days_since_creation < 1000` returns zero rows on current data, since all customers were created years ago.)

## Part 2: Add Widgets to build the Dashboard

You'll build the **first widget (Total MRR) by hand** so you can see how a widget maps to a dataset. After that, use **Genie Code** to build each remaining widget from a single prompt — with the manual steps available as an optional fallback.

### 2.1: Add a Metric — "Total MRR ($)" *(build this one manually)*

Click the **Canvas** tab, then the **Widget** icon at the bottom to add a widget. In the **Widget** panel on the right-hand side, set the following:

- **Title Checkbox**: Enabled
- **Title Value**: Total MRR ($)
- **Dataset**: Churn - Total MRR - Universal
- **Visualization**: Counter
- **Value**: MRR
  - **Transform**: SUM
- **Value Format**: Custom
  - **Type**: $
  - **Abbreviation**: None
  - **Decimal Places**: Max, 0
  - **Group Separator**: Checked

![](./Images/aibi_dashboard/Dashboard_Add_Widget1.png)

### 2.2: Add the Bar chart — "Predicted as Churn - Customer Tenure"

**Recommended — build it with Genie Code.** Open the **Genie Code** panel (Agent mode) and paste:

> Using the **Churn - Predictions - Universal** dataset, add a **bar chart** titled **"Predicted as Churn - Customer Tenure"**. Put **days_since_creation** on the **X-axis** (continuous scale), the **SUM of customers** on the **Y-axis**, and **color the bars by churn** (categorical scale). Give the two churn groups distinct colors.

<details>
<summary><b>Optional:</b> build it manually instead</summary>

In the **Widget** panel on the right-hand side, set:

- **Title Checkbox**: Enabled
- **Title Value**: Predicted as Churn - Customer Tenure
- **Dataset**: Churn - Predictions - Universal
- **Visualization**: Bar
- **X axis**: days_since_creation — **Scale Type**: Continuous, **Transform**: None
- **Y axis**: SUM(customers)
- **Color**: churn — **Scale Type**: Categorical, **Transform**: None
- Adjust the bar colors to match your color scheme
</details>

### 2.3: Add the Pie chart — "At Risk Customers - Have Device Protection"

**Recommended — build it with Genie Code.** Open the **Genie Code** panel (Agent mode) and paste:

> Using the **Churn - Customer Breakdown - Universal** dataset, add a **pie chart** titled **"At Risk Customers - Have Device Protection"**. Use the **SUM of users_count** as the **angle**, **color the slices by canal**, and **enable data labels**.

<details>
<summary><b>Optional:</b> build it manually instead</summary>

In the **Widget** panel on the right-hand side, set:

- **Title Checkbox**: Enabled
- **Title Value**: At Risk Customers - Have Device Protection
- **Dataset**: Churn - Customer Breakdown - Universal
- **Visualization**: Pie
- **Angle**: SUM(users_count)
- **Color**: canal
- **Labels**: Enabled
</details>

##Part 3: Add Filter


### 3.1: Add the Country filter

**Recommended — build it with Genie Code.** Open the **Genie Code** panel (Agent mode) and paste:

> Add a **filter widget** titled **"Country Filter"** on the **country** field, configured to **allow multiple values**. Connect it to the **country** field of **every dataset** on the dashboard so it filters all widgets at once.

<details>
<summary><b>Optional:</b> build it manually instead</summary>

In the **Widget** panel on the right-hand side, set:

- Click the **Canvas** tab, then the **Filter** button at the bottom, and place the filter widget
- **Title**: Country Filter
- **Filter**: Multiple values
- **Field**: select any dataset → country (add the **country** field from *each* dataset to associate them all)
</details>

##Part 4: Publish and Embed Dashboards



### 4.1 Publish Dashboard

Click on the Publish button as shown below.


![](./Images/aibi_dashboard/Dashboard_Publish_1_1.png)


### 4.2 Publish Dashboard

Review the information and click on the Publish button as shown below.


![](./Images/aibi_dashboard/Dashboard_Publish_2.png)

### 4.3 Scheduling Dashboard
Now that we have our dashboard published. Let's schedule it so it refreshes at set periods. Click Schedule and create a schedule to happen every 5 minutes.

![](./Images/aibi_dashboard/Dashboard_Schedule_1_0.png)


### 4.4 Subscribe to the Schedule for Snapshots
Click Schedule, you should now see the schedule you just created. Click Subscribe. This will send a snapshot of the dashboard as a pdf to your email, or any email subscribed to the dashboard schedule. 

![](./Images/aibi_dashboard/Dashboard_Schedule_1_1.png)

If you would like to add subscribes that are not yourself, first click edit on the schedule. In the edit screen, there will be a tab labeled **Subscribers** which will allow you to enter other users.

### 4.5 Check your email for Snapshots

If the schedule has been tirggered. You should have recieved an email with a snapshot of the dashboard. Attached to this email is a pdf version that makes for easier viewing.

![](./Images/aibi_dashboard/Dashboard_Schedule_1_2.png)

### 4.6 Delete the Schedule
We will delete the schedule to avoid running uncessary compute and stop the incoming emails. Click Schedule -> Kebab Icon on the Schedule -> Delete to delte the recurring schedule and emails.

![](./Images/aibi_dashboard/Dashboard_Schedule_1_3.png)

##Conclusion

We saw how we can use Databricks AIBI to quickly create a Dashboard on our freshest data and enable data-driven decision-making for everyone. If you would like, feel free to continue this lab. The next session will fully build out the remainder of this dashboard. Lastly, it will walk you through how you can  embed it.


## Part 5 : Extended Version



### 5.1: Add the following datasets

Add each dataset below the same way as Part 1 ("Create from SQL"), then continue building widgets.

| Dataset Name | SQL Query |
| --- | --- |
| Churn - Monthly Revenue Risk - Universal | `SELECT sum(amount)/100 as MRR_at_risk, country FROM data_pioneers.c360.churn_orders o LEFT JOIN data_pioneers.c360.churn_users cu ON o.user_id = cu.user_id WHERE month(to_timestamp(o.creation_date, 'MM-dd-yyyy HH:mm:ss')) = (SELECT max(month(to_timestamp(creation_date, 'MM-dd-yyyy HH:mm:ss'))) FROM data_pioneers.c360.churn_orders) AND o.user_id IN (SELECT user_id FROM data_pioneers.c360.churn_users WHERE churn = 1) GROUP BY country` |
| Churn - Subscriptions based on Internet Service | `SELECT platform, churn, country, count(*) as event_count FROM data_pioneers.c360.churn_app_events INNER JOIN data_pioneers.c360.churn_users USING (user_id) WHERE platform IS NOT NULL GROUP BY platform, churn, country` |
| Churn - Predicted Churn Percent Payment Method - Universal | `SELECT p.country, p.churn, count(*) as customers FROM data_pioneers.c360.churn_features p GROUP BY p.country, p.churn` |
| Churn - At-Risk Customers - Universal | `SELECT country, count(*)/12 as at_risk FROM data_pioneers.c360.churn_users WHERE churn = 1 GROUP BY country` |
| Churn - Predicted to Churn - Universal | `SELECT user_id, churn, country FROM data_pioneers.c360.churn_users WHERE churn = 1` |

### 5.2: Add the Counter — "Predicted Churn - Monthly Revenue Risk ($)"

**Recommended — build it with Genie Code.** Open the **Genie Code** panel (Agent mode) and paste:

> In the **Churn - Monthly Revenue Risk - Universal** dataset, add a **Counter** widget titled **"Predicted Churn - Monthly Revenue Risk ($)"**. Use **MRR_at_risk** as the value with a **SUM** aggregation, formatted as **currency ($)** with **0 decimal places**.

<details>
<summary><b>Optional:</b> build it manually instead</summary>

In the **Widget** panel on the right-hand side, set:

- **Title Checkbox**: Enabled
- **Title Value**: Predicted Churn - Monthly Revenue Risk ($)
- **Dataset**: Churn - Monthly Revenue Risk - Universal
- **Visualization**: Counter
- **Value**: MRR_at_risk — **Transform**: SUM
  - **Format**: Custom → **Type**: $
</details>

### 5.3: Add the Counter — "At-Risk Customers"

**Recommended — build it with Genie Code.** Open the **Genie Code** panel (Agent mode) and paste:

> In the **Churn - At-Risk Customers - Universal** dataset, add a **Counter** widget titled **"At-Risk Customers"**. Use **at_risk** as the value with a **SUM** aggregation, formatted as a **number with 0 decimal places** (no decimals).

<details>
<summary><b>Optional:</b> build it manually instead</summary>

In the **Widget** panel on the right-hand side, set:

- **Title Checkbox**: Enabled
- **Title Value**: At-Risk Customers
- **Dataset**: Churn - At-Risk Customers - Universal
- **Visualization**: Counter
- **Value**: at_risk — **Transform**: SUM
- **Format**: remove decimals (Decimal Places: 0)
</details>

### 5.4: Add the Bar chart — "Percent Predicted Churn - Payment Method"

**Recommended — build it with Genie Code.** Open the **Genie Code** panel (Agent mode) and paste:

> Using the **Churn - Predicted Churn Percent Payment Method - Universal** dataset, add a **bar chart** titled **"Percent Predicted Churn - Payment Method"**. Put **churn** on the **X-axis** (categorical scale), the **SUM of customers** on the **Y-axis**, and **color the bars by country**.

<details>
<summary><b>Optional:</b> build it manually instead</summary>

In the **Widget** panel on the right-hand side, set:

- **Title Checkbox**: Enabled
- **Title Value**: Percent Predicted Churn - Payment Method
- **Dataset**: Churn - Predicted Churn Percent Payment Method - Universal
- **Visualization**: Bar
- **X axis**: churn — **Scale Type**: Categorical
- **Y axis**: customers — **Transform**: SUM
- **Color**: country
</details>

### 5.5: Add the Table — "Customers Predicted to Churn"

**Recommended — build it with Genie Code.** Open the **Genie Code** panel (Agent mode) and paste:

> Using the **Churn - Predicted to Churn - Universal** dataset, add a **table** widget titled **"Customers Predicted to Churn"** showing all columns (**user_id, churn, country**).

<details>
<summary><b>Optional:</b> build it manually instead</summary>

In the **Widget** panel on the right-hand side, set:

- **Title Checkbox**: Enabled
- **Title Value**: Customers Predicted to Churn
- **Dataset**: Churn - Predicted to Churn - Universal
- **Visualization**: Table
- **Columns**: Show/hide all
</details>

### 5.6: Add the Bar chart — "Churn - Subscriptions based on Internet Service"

**Recommended — build it with Genie Code.** Open the **Genie Code** panel (Agent mode) and paste:

> Using the **Churn - Subscriptions based on Internet Service** dataset, add a **bar chart** titled **"Churn - Subscriptions based on Internet Service"**. Put the **SUM of event_count** on the **X-axis** and **platform** on the **Y-axis** (horizontal bars), and **color by churn** (categorical scale).

<details>
<summary><b>Optional:</b> build it manually instead</summary>

In the **Widget** panel on the right-hand side, set:

- **Title Checkbox**: Enabled
- **Title Value**: Churn - Subscriptions based on Internet Service
- **Dataset**: Churn - Subscriptions based on Internet Service
- **Visualization**: Bar
- **X axis**: SUM(event_count)
- **Y axis**: platform
- **Color**: churn — **Scale Type**: Categorical, **Transform**: None
</details>

### 5.7: Add Country to the filter

**Recommended — build it with Genie Code.** Open the **Genie Code** panel (Agent mode) and paste:

> Update the **Country Filter** so it also covers the newly added datasets — connect the filter's **country** field to the **country** column of each new dataset.

<details>
<summary><b>Optional:</b> build it manually instead</summary>

In the **Widget** panel on the right-hand side, set:

- Go to the **Country Filter**
- Add the **country** attribute from each of the newly added datasets to associate them with the filter
</details>

## Part 6 : Embedding the Dashboard


### 6.1 Create Embedding

Click on the Share button to generate embedding iframe that can be used to embed the dashboard in applications


![](./Images/aibi_dashboard/Dashboard_Embed_1_1.png)


### 6.2 Create Embedding

Click on Embed Dashboard button to generate the iframe url as shown below.


![](./Images/aibi_dashboard/Dashboard_Embed_2.png)


### 6.3 Create Embedding

The generated URL can now be used to embed the dashboard in applications.


![](./Images/aibi_dashboard/Dashboard_Embed_3.png)

## 🎉 Recap & where to go next

You built a Customer Churn dashboard end-to-end — the first widget by hand, the rest with **Genie Code**:

| You built... | In... | Why it matters |
| --- | --- | --- |
| The dashboard + SQL datasets | Part 1 | The governed data layer every widget draws from. |
| The Total MRR counter (by hand), then the rest with Genie Code | Part 2 | You learned how a widget maps to a dataset, then built the rest from plain English. |
| A cross-widget **Country** filter | Part 3 | One control that filters the entire dashboard. |
| A **published, scheduled** dashboard | Part 4 | Fresh insights delivered automatically. |
| **Extended** widgets — revenue-at-risk, payment-method churn, tables | Part 5 | Deeper churn analysis, still built conversationally. |
| An **embedded** dashboard | Part 6 | Insights surfaced inside the apps your users already use. |

**The big idea:** learn the mechanics once by hand, then describe each widget to **Genie Code** — the same natural-language, AI-assisted workflow you'll use across Databricks.

**Where to go next**
- Describe a brand-new widget of your own to Genie Code.
- Attach an **AI/BI Genie Space** (Lab 3) so users can ask follow-up questions on this dashboard.